In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import logging
import os
import sys

# Set up project path for Jupyter notebook
project_root = Path().resolve().parent  # Assuming notebook is in tests directory
sys.path.append('')

# Set up logging
logging.basicConfig(level=logging.INFO)

# Import the AtmosphericCorrection class
from src.core.atmospheric import AtmosphericCorrection

def create_sample_data(size=512):
    """Create synthetic satellite imagery data"""
    np.random.seed(42)
    
    # Create base pattern
    x, y = np.mgrid[0:size, 0:size]
    base_feature = np.sin(x/50) + np.cos(y/50)
    
    # Add atmospheric effects for each band
    red_band = base_feature + np.random.normal(0.2, 0.05, (size, size))
    green_band = base_feature + np.random.normal(0.15, 0.05, (size, size))
    blue_band = base_feature + np.random.normal(0.1, 0.05, (size, size))
    
    # Stack bands and normalize
    data = np.stack([red_band, green_band, blue_band], axis=-1)
    data = (data - data.min()) / (data.max() - data.min())
    
    return data

def visualize_correction(original, corrected, method):
    """Visualize original and corrected images side by side"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    
    # Original image
    im1 = ax1.imshow(original)
    ax1.set_title('Original Image')
    plt.colorbar(im1, ax=ax1)
    
    # Corrected image
    im2 = ax2.imshow(corrected)
    ax2.set_title(f'Corrected Image ({method})')
    plt.colorbar(im2, ax=ax2)
    
    plt.tight_layout()
    plt.show()

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

print(f"Project root: {project_root}")
print(f"Output directory: {output_dir}")

# Create sample data
print("Creating sample satellite data...")
data = create_sample_data()

# Create metadata
metadata = {
    "sun_elevation": 45.0,
    "sun_azimuth": 180.0,
    "acquisition_date": datetime.now().strftime("%Y-%m-%d"),
    "altitude": 705,  # typical satellite altitude in km
    "latitude": 0.0,
    "longitude": 0.0
}

# Initialize atmospheric correction
corrector = AtmosphericCorrection()

# Test all correction methods
methods = ["dos", "6s", "modtran"]

for method in methods:
    print(f"\nApplying {method.upper()} correction...")
    try:
        # Apply correction
        corrected_data, updated_metadata = corrector.correct_image(
            data=data,
            metadata=metadata.copy(),
            method=method
        )
        
        # Show visualization
        visualize_correction(data, corrected_data, method)
        
        # Print statistics
        print(f"\n{method.upper()} Correction Results:")
        print("-" * 30)
        print(f"Original data range: [{data.min():.3f}, {data.max():.3f}]")
        print(f"Corrected data range: [{corrected_data.min():.3f}, {corrected_data.max():.3f}]")
        print(f"Mean correction: {(corrected_data - data).mean():.3f}")
        
        # Print correction details
        correction_info = updated_metadata.get("atmospheric_correction", {})
        print("\nCorrection Details:")
        for key, value in correction_info.items():
            if key != "timestamp":
                print(f"{key}: {value}")
        
    except Exception as e:
        print(f"Error applying {method} correction: {e}")
        continue

print("\nProcessing complete!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import logging
import os
import sys
from PIL import Image

# Set up project path for Jupyter notebook
project_root = Path().resolve().parent  # Assuming notebook is in tests directory
sys.path.append('/home/jaya/synthetic-satellite')

# Set up logging
logging.basicConfig(level=logging.INFO)

# Import the AtmosphericCorrection class
from src.core.atmospheric import AtmosphericCorrection

def load_image(image_path='a2.png'):
    """Load and preprocess the input image"""
    # Load image
    img = Image.open(image_path)
    
    # Convert to numpy array and normalize
    data = np.array(img).astype(np.float32)
    
    # Normalize to 0-1 range
    data = data / 255.0
    
    # Ensure we have 3 channels (RGB)
    if len(data.shape) == 2:  # If grayscale
        data = np.stack([data] * 3, axis=-1)
    elif data.shape[-1] == 4:  # If RGBA
        data = data[:, :, :3]  # Take only RGB channels
    
    return data

def visualize_correction(original, corrected, method):
    """Visualize original and corrected images side by side"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    
    # Original image
    im1 = ax1.imshow(original)
    ax1.set_title('Original Image')
    plt.colorbar(im1, ax=ax1)
    
    # Corrected image
    im2 = ax2.imshow(corrected)
    ax2.set_title(f'Corrected Image ({method})')
    plt.colorbar(im2, ax=ax2)
    
    plt.tight_layout()
    plt.show()
    
    # Show difference map
    plt.figure(figsize=(8, 6))
    diff = corrected - original
    im3 = plt.imshow(np.mean(diff, axis=2), cmap='RdBu')
    plt.colorbar(im3, label='Difference')
    plt.title(f'Correction Difference Map ({method})')
    plt.show()

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

print(f"Project root: {project_root}")
print(f"Output directory: {output_dir}")

# Load the actual image
print("Loading image data...")
data = load_image('a2.png')
print(f"Image shape: {data.shape}")

# Create metadata
metadata = {
    "sun_elevation": 45.0,
    "sun_azimuth": 180.0,
    "acquisition_date": datetime.now().strftime("%Y-%m-%d"),
    "altitude": 705,  # typical satellite altitude in km
    "latitude": 0.0,
    "longitude": 0.0
}

# Initialize atmospheric correction
corrector = AtmosphericCorrection()

# Test all correction methods
methods = ["dos", "6s", "modtran"]

for method in methods:
    print(f"\nApplying {method.upper()} correction...")
    try:
        # Apply correction
        corrected_data, updated_metadata = corrector.correct_image(
            data=data,
            metadata=metadata.copy(),
            method=method
        )
        
        # Show visualization
        visualize_correction(data, corrected_data, method)
        
        # Save corrected image
        corrected_image = Image.fromarray((corrected_data * 255).astype(np.uint8))
        corrected_image.save(output_dir / f"{method}_corrected.png")
        
        # Print statistics
        print(f"\n{method.upper()} Correction Results:")
        print("-" * 30)
        print(f"Original data range: [{data.min():.3f}, {data.max():.3f}]")
        print(f"Corrected data range: [{corrected_data.min():.3f}, {corrected_data.max():.3f}]")
        print(f"Mean correction: {(corrected_data - data).mean():.3f}")
        
        # Print correction details
        correction_info = updated_metadata.get("atmospheric_correction", {})
        print("\nCorrection Details:")
        for key, value in correction_info.items():
            if key != "timestamp":
                print(f"{key}: {value}")
        
    except Exception as e:
        print(f"Error applying {method} correction: {e}")
        continue

print("\nProcessing complete!")